# ema-second-moment — faded example 1: EMA v Update: In-Place List Step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-second-moment`. Running the beacon reports progress on the `Optimizer: Adam EMA second moment` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA second moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-second-moment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-second-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA second moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Adam's second-moment update `v = beta2 * v + (1 - beta2) * g**2` must be applied in-place to each buffer tensor. Using `v.copy_(...)` mutates the tensor at its existing memory address so any external reference (such as one held by the optimizer) automatically sees the new value. Rebinding with `v = ...` would silently break this contract.

## Faded exercise 1

Implement `ema_v_step(v_list, grad_list, beta2)` that performs one Adam second-moment update across a list of (v, g) buffer/gradient pairs.

For each pair, apply `v = beta2 * v + (1 - beta2) * g.pow(2)` **in-place** using `v.copy_()`, then append the updated tensor to the output list. Return the output list.

**Fill in:** Apply the EMA formula in-place to v using v.copy_(), storing the result of beta2*v + (1-beta2)*g.pow(2).

In [ ]:
import torch as t
import torch.nn as nn

def ema_v_step(v_list, grad_list, beta2):
    out = []
    for v, g in zip(v_list, grad_list):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
        out.append(v)
    return out


def _test():
    import torch as t
    t.manual_seed(99)
    v_list = [t.zeros(3), t.zeros(3)]
    grad_list = [t.tensor([1.0, 2.0, 3.0]), t.tensor([4.0, 0.5, 1.5])]
    ptrs = [v.data_ptr() for v in v_list]
    beta2 = 0.9
    result = ema_v_step(v_list, grad_list, beta2)
    # Check in-place: same data pointers
    assert [r.data_ptr() for r in result] == ptrs
    # Check values: v = 0.9*0 + 0.1*g^2
    expected_0 = 0.1 * t.tensor([1.0, 4.0, 9.0])
    expected_1 = 0.1 * t.tensor([16.0, 0.25, 2.25])
    assert t.allclose(result[0], expected_0, atol=1e-6)
    assert t.allclose(result[1], expected_1, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def ema_v_step(v_list, grad_list, beta2):
    out = []
    for v, g in zip(v_list, grad_list):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
        out.append(v)
    return out
```
</details>